# Step 0: Verify ViFactCheck Labels

Run `recover_labels.py` to inject labels from HuggingFace into crawled JSON,
then verify the label distribution per split.

**Run this once** (or whenever you re-crawl data).

- Input: `../data/json/news_data_vifactcheck_*_cleaned.json` (raw crawled)
- Output: `../data/json/labeled_nei_as_real/*_labeled.json` (with labels)

In [1]:
import subprocess, sys, os
from pathlib import Path
from collections import Counter

os.chdir(Path().absolute().parent)  # notebooks/ dir
print(f"Working dir: {os.getcwd()}")

Working dir: g:\My Drive\Thesis_Final\fake-new-detection\notebooks


In [2]:
# Generate labels with binary_nei_as_real strategy
# NEI (Not Enough Information) treated as Real -> better class balance
subprocess.run([
    sys.executable, "recover_labels.py",
    "--strategy", "binary_nei_as_real",
    "--input-dir", "./data/json",
    "--output-dir", "./data/json/labeled_nei_as_real"
], check=True)
print("\nLabels generated.")

CalledProcessError: Command '['c:\\Users\\tungh\\.conda\\envs\\fake_news\\python.exe', 'recover_labels.py', '--strategy', 'binary_nei_as_real', '--input-dir', './data/json', '--output-dir', './data/json/labeled_nei_as_real']' returned non-zero exit status 1.

In [ ]:
# Verify label distribution
import json

data_dir = Path("./data/json/labeled_nei_as_real")

for split in ["train", "dev", "test"]:
    path = data_dir / f"news_data_vifactcheck_{split}_labeled.json"
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    labels = [d["label"] for d in data]
    dist = Counter(labels)
    total = len(labels)
    print(f"{split}: {total} samples")
    for lbl in sorted(dist):
        name = "Real" if lbl == 0 else "Fake"
        print(f"  {name} (label={lbl}): {dist[lbl]} ({dist[lbl]/total*100:.1f}%)")
    
    # Check images
    has_img = sum(1 for d in data if d.get("images") and len(d["images"]) > 0)
    print(f"  With images: {has_img}/{total}")
    print()

print("Expected: dev set ~49% real / ~51% fake (nearly balanced)")
print("If correct, proceed to Step 1: Preprocessing.")

train: 4509 samples
  Real (label=0): 341 (7.6%)
  Fake (label=1): 4168 (92.4%)
  With images: 3232/4509

dev: 634 samples
  Real (label=0): 311 (49.1%)
  Fake (label=1): 323 (50.9%)
  With images: 451/634

test: 1282 samples
  Real (label=0): 496 (38.7%)
  Fake (label=1): 786 (61.3%)
  With images: 935/1282

Expected: dev set ~49% real / ~51% fake (nearly balanced)
If correct, proceed to Step 1: Preprocessing.


: 